# 🏛️ POC 9: Deep Historical Stress-Testing Across 26+ Years (2000–2026)

**File**: [`research/notebooks/algo-alpha-execution/09_deep_historical_backtest_2000_2026.ipynb`](file:///c:/Users/honza/Desktop/projects/stock-analysis/research/notebooks/algo-alpha-execution/09_deep_historical_backtest_2000_2026.ipynb)  
**Historical Backtest Horizon**: **January 2000 - August 2026 (26.6 Years / 6,600+ Daily Trading Sessions)**  
**Historical Crisis Regimes Evaluated**:
1. **2000–2002 Dot-Com Bubble Crash**
2. **2007–2009 Global Financial Crisis (Lehman Collapse)**
3. **2010 Flash Crash & 2011 US Sovereign Debt Downgrade**
4. **2015–2016 Global Commodity Slump**
5. **2018 Volmageddon & Q4 Sell-off**
6. **2020 COVID-19 Flash Crash**
7. **2022 Inflation & Rate-Hike Bear Market**
8. **2023–2026 AI Secular Expansion**

**Primary Models & Benchmarks Focused**:
- 🏆 **Unified "One-for-All" Engine (Concentrated Top 10 Active)**: Confluence Dynamic Sizing, Trailing ATR Stops ($2.5 \cdot \text{ATR}_{14}$), and $2.0\sigma$ Macro Volatility Guard.
- 🤖 **Baseline Naive XGBoost (Top 100)**: Active Top 100 walk-forward equal-weighted.
- 📈 **S&P 500 Index (`SPY` Benchmark)**: The official US market benchmark across all 26.6 years.

---

### Executive Summary & Institutional Synthesis
This notebook subjects our algorithmic engines to every major financial crisis over the last quarter-century with zero lookahead bias:
- **Survival Through 2008 & 2000**: How trailing ATR stops and macro volatility cash buffers protect capital during -50% market crashes.
- **26-Year Compounding Multiplier**: The generational wealth-creation difference between active asymmetric allocation vs. passive buy-and-hold.

## 1. Setup, Configuration & Dependencies

In [1]:
import os
import sys
import datetime
from datetime import timedelta
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import yfinance as yf
from tqdm.auto import tqdm

# Robust project root discovery
current_dir = os.path.abspath(os.getcwd())
while current_dir and not os.path.exists(os.path.join(current_dir, "src")):
    parent = os.path.dirname(current_dir)
    if parent == current_dir:
        break
    current_dir = parent

PROJECT_ROOT = current_dir
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.config import DATA_DIR, INITIAL_CAPITAL

LOCAL_DATA_DIR = os.path.join(PROJECT_ROOT, "research", "notebooks", "algo-alpha-execution", "data", "fetched")
if not os.path.exists(LOCAL_DATA_DIR):
    LOCAL_DATA_DIR = os.path.join(PROJECT_ROOT, "data", "fetched")

print(f"📁 Project Root: {PROJECT_ROOT}")
print(f"📁 Local Data Directory: {LOCAL_DATA_DIR}")
print(f"💰 Initial Capital: ${INITIAL_CAPITAL}")
print(f"⏳ Backtest Scope: January 2000 - August 2026 (26.6 Years)")

📁 Project Root: c:\Users\honza\Desktop\projects\stock-analysis
📁 Local Data Directory: c:\Users\honza\Desktop\projects\stock-analysis\research\notebooks\algo-alpha-execution\data\fetched
💰 Initial Capital: $100.0
⏳ Backtest Scope: January 2000 - August 2026 (26.6 Years)


## 2. Ingesting 26-Year Historical Predictions, Market Prices & ATR Matrix

In [2]:
def load_deep_dataset():
    preds_path = os.path.join(LOCAL_DATA_DIR, "deep_historical_2000_2026_predictions_poc.xlsx")
    df_preds = pd.read_excel(preds_path)
    df_preds['date'] = pd.to_datetime(df_preds['date'])
    all_tickers = sorted(df_preds['ticker'].unique())
    print(f"✅ Loaded {len(df_preds)} prediction records across {len(all_tickers)} tickers over 2000-2026!")
    
    unique_tickers = all_tickers + ['SPY']
    min_date = (df_preds['date'].min() - timedelta(days=60)).strftime('%Y-%m-%d')
    max_date = (df_preds['date'].max() + timedelta(days=10)).strftime('%Y-%m-%d')
    
    print(f"📈 Downloading OHLC market data from 1999 to 2026 for {len(unique_tickers)} tickers...")
    ohlc = yf.download(unique_tickers, start=min_date, end=max_date, auto_adjust=True, progress=False)
    
    close_p = ohlc['Close']
    high_p = ohlc['High']
    low_p = ohlc['Low']
    
    close_p.index = pd.to_datetime(close_p.index).tz_localize(None)
    high_p.index = pd.to_datetime(high_p.index).tz_localize(None)
    low_p.index = pd.to_datetime(low_p.index).tz_localize(None)
    
    # Compute ATR(14)
    atr_dict = {}
    for t in all_tickers:
        if t in close_p.columns and t in high_p.columns and t in low_p.columns:
            c = close_p[t]
            h = high_p[t]
            l = low_p[t]
            prev_c = c.shift(1)
            tr = pd.concat([h - l, (h - prev_c).abs(), (l - prev_c).abs()], axis=1).max(axis=1)
            atr_dict[t] = tr.ewm(alpha=1/14, adjust=False).mean()
    df_atr = pd.DataFrame(atr_dict)
    
    # Macro Volatility Filter across 26 years
    spy_ret = close_p['SPY'].pct_change()
    ewma_lam = 1.0 - (2.0 / 21.0)
    spy_ewma_var = (spy_ret**2).ewm(alpha=(1 - ewma_lam), adjust=False).mean()
    spy_ewma_vol = np.sqrt(spy_ewma_var) * np.sqrt(252)
    spy_vol_ma = spy_ewma_vol.rolling(60).mean()
    spy_vol_std = spy_ewma_vol.rolling(60).std()
    spy_vol_zscore = (spy_ewma_vol - spy_vol_ma) / (spy_vol_std + 1e-9)
    
    return df_preds, close_p, df_atr, spy_vol_zscore, all_tickers

df_predictions, close_prices, df_atr_matrix, macro_vol_z, universe_tickers = load_deep_dataset()
all_sim_dates = sorted(list(set(df_predictions['date'].unique()) & set(close_prices.index)))
print(f"✅ Total 26-Year Simulation Trading Days: {len(all_sim_dates)} ({all_sim_dates[0].strftime('%Y-%m-%d')} to {all_sim_dates[-1].strftime('%Y-%m-%d')})")

✅ Loaded 845222 prediction records across 129 tickers over 2000-2026!
📈 Downloading OHLC market data from 1999 to 2026 for 130 tickers...


✅ Total 26-Year Simulation Trading Days: 6577 (2000-07-03 to 2026-08-27)


C:\Users\honza\AppData\Local\Temp\ipykernel_23280\1904186549.py:36: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  spy_ret = close_p['SPY'].pct_change()


## 3. Parametric Strategy Simulation Across 2000–2026

In [3]:
def simulate_unified_alpha_engine(
    preds_df, prices_df, atr_df, macro_z, all_dates,
    base_cap=0.08, max_confluence_cap=0.20, atr_multiplier=2.5, rebalance_days=25, z_threshold=2.0, active_n=10, transaction_cost_bps=15
):
    fee_rate = transaction_cost_bps / 10000.0
    cash = INITIAL_CAPITAL
    active_positions = {}
    history = []
    days_since_rebalance = rebalance_days
    
    for d in all_dates:
        p_now = prices_df.loc[d]
        atr_now = atr_df.loc[d] if d in atr_df.index else None
        z_curr = macro_z.loc[d] if d in macro_z.index else 0.0
        
        stopped_out = []
        for t, pos in list(active_positions.items()):
            if t in p_now and pd.notna(p_now[t]):
                price_curr = p_now[t]
                atr_curr = atr_now[t] if (atr_now is not None and t in atr_now and pd.notna(atr_now[t])) else (price_curr * 0.03)
                if price_curr > pos['highest_price']:
                    pos['highest_price'] = price_curr
                    pos['stop_price'] = max(pos['stop_price'], price_curr - (atr_multiplier * atr_curr))
                if price_curr <= pos['stop_price']:
                    cash += pos['shares'] * price_curr * (1.0 - fee_rate)
                    stopped_out.append(t)
        for t in stopped_out:
            del active_positions[t]
            
        if days_since_rebalance >= rebalance_days:
            days_since_rebalance = 0
            is_vol_spike = (pd.notna(z_curr) and z_curr > z_threshold)
            cash_buffer_ratio = 0.30 if is_vol_spike else 0.0
            
            day_preds = preds_df[preds_df['date'] == d].copy()
            pos_preds = day_preds[day_preds['predicted_return_multimodal'] > 0.0]
            if pos_preds.empty:
                pos_preds = day_preds
                
            if not pos_preds.empty:
                selected = pos_preds.sort_values('predicted_return_multimodal', ascending=False).head(active_n)
                conf_score = selected['confluence_score'] if 'confluence_score' in selected.columns else 0.0
                caps = base_cap + (conf_score / 6.0) * (max_confluence_cap - base_cap)
                caps = caps.clip(lower=base_cap, upper=max_confluence_cap)
                inv_vols = 1.0 / selected['ewma_volatility'].clip(lower=0.05)
                raw_weights = inv_vols / inv_vols.sum()
                bounded_weights = np.minimum(raw_weights, caps)
                final_weights = (bounded_weights / bounded_weights.sum()) * (1.0 - cash_buffer_ratio)
                target_alloc = dict(zip(selected['ticker'], final_weights))
            else:
                target_alloc = {}
                
            for t in list(active_positions.keys()):
                if t not in target_alloc:
                    if t in p_now and pd.notna(p_now[t]):
                        cash += active_positions[t]['shares'] * p_now[t] * (1.0 - fee_rate)
                    del active_positions[t]
                    
            total_fund = cash + sum(pos['shares'] * p_now[t] for t, pos in active_positions.items() if t in p_now)
            cash = total_fund * cash_buffer_ratio
            investable = total_fund * (1.0 - cash_buffer_ratio)
            
            for t, w in target_alloc.items():
                if t in p_now and pd.notna(p_now[t]) and p_now[t] > 0:
                    price_curr = p_now[t]
                    atr_curr = atr_now[t] if (atr_now is not None and t in atr_now and pd.notna(atr_now[t])) else (price_curr * 0.03)
                    shares = (investable * (w / (1.0 - cash_buffer_ratio + 1e-9)) * (1.0 - fee_rate)) / price_curr
                    active_positions[t] = {
                        'shares': shares,
                        'entry_price': price_curr,
                        'highest_price': price_curr,
                        'stop_price': price_curr - (atr_multiplier * atr_curr),
                        'weight': w
                    }
                    
        days_since_rebalance += 1
        portfolio_val = cash + sum(pos['shares'] * p_now[t] for t, pos in active_positions.items() if t in p_now and pd.notna(p_now[t]))
        history.append({'date': d, 'portfolio_value': portfolio_val})
        
    return pd.DataFrame(history)

def simulate_baseline_top100_strategy(
    preds_df, prices_df, all_dates,
    rebalance_days=25, active_n=100, transaction_cost_bps=15
):
    fee_rate = transaction_cost_bps / 10000.0
    cash = INITIAL_CAPITAL
    active_positions = {}
    history = []
    days_since_rebalance = rebalance_days
    
    for d in all_dates:
        p_now = prices_df.loc[d]
        
        if days_since_rebalance >= rebalance_days:
            days_since_rebalance = 0
            day_preds = preds_df[preds_df['date'] == d].copy()
            pos_preds = day_preds[day_preds['predicted_return_baseline'] > 0.0]
            if pos_preds.empty:
                pos_preds = day_preds
                
            if not pos_preds.empty:
                selected = pos_preds.sort_values('predicted_return_baseline', ascending=False).head(active_n)
                weights = [1.0 / len(selected)] * len(selected)
                target_alloc = dict(zip(selected['ticker'], weights))
            else:
                target_alloc = {}
                
            for t in list(active_positions.keys()):
                if t not in target_alloc:
                    if t in p_now and pd.notna(p_now[t]):
                        cash += active_positions[t]['shares'] * p_now[t] * (1.0 - fee_rate)
                    del active_positions[t]
                    
            total_fund = cash + sum(pos['shares'] * p_now[t] for t, pos in active_positions.items() if t in p_now)
            cash = 0.0
            
            for t, w in target_alloc.items():
                if t in p_now and pd.notna(p_now[t]) and p_now[t] > 0:
                    shares = (total_fund * w * (1.0 - fee_rate)) / p_now[t]
                    active_positions[t] = {'shares': shares, 'entry_price': p_now[t], 'weight': w}
                    
        days_since_rebalance += 1
        portfolio_val = cash + sum(pos['shares'] * p_now[t] for t, pos in active_positions.items() if t in p_now and pd.notna(p_now[t]))
        history.append({'date': d, 'portfolio_value': portfolio_val})
        
    return pd.DataFrame(history)

print("🚀 Running 2000-2026 Deep Historical Backtest (26.6 Years)...")

# 1. Concentrated Flagship Alpha Engine (Active Top 10)
df_strat_conc = simulate_unified_alpha_engine(df_predictions, close_prices, df_atr_matrix, macro_vol_z, all_sim_dates, base_cap=0.08, max_confluence_cap=0.20, rebalance_days=25, active_n=10)

# 2. Broad Baseline Naive XGBoost (Active Top 100)
df_strat_base = simulate_baseline_top100_strategy(df_predictions, close_prices, all_sim_dates, rebalance_days=25, active_n=100)

# 3. S&P 500 Benchmark (SPY)
sim_dates = df_strat_conc['date']
spy_prices = close_prices['SPY'].loc[close_prices.index.isin(sim_dates)]
spy_norm = (spy_prices / spy_prices.iloc[0]) * INITIAL_CAPITAL

df_master_eval = pd.DataFrame({
    'date': sim_dates,
    'Strategy_Unified_Concentrated_Top10': df_strat_conc['portfolio_value'].values,
    'Strategy_Baseline_XGBoost_Top100': df_strat_base['portfolio_value'].values,
    'Benchmark_SPY_SP500': spy_norm.values
})

df_master_eval.head(10)

🚀 Running 2000-2026 Deep Historical Backtest (26.6 Years)...


,date,Strategy_Unified_Concentrated_Top10,Strategy_Baseline_XGBoost_Top100,Benchmark_SPY_SP500
0,2000-07-03,99.850000,99.850000,100.000000
1,2000-07-05,94.786954,98.410045,98.196521
2,2000-07-06,96.503748,98.855844,98.960370
3,2000-07-07,97.388602,99.884188,100.551718
4,2000-07-10,99.048261,100.319912,100.381978
5,2000-07-11,100.000013,101.352043,100.594151
6,2000-07-12,103.093707,101.903089,101.251909
7,2000-07-13,103.077961,101.776235,101.697463
8,2000-07-14,105.794532,102.843378,102.694709
9,2000-07-17,106.984125,102.518399,102.524977


## 4. Quantitative Analytics & 26-Year Performance Matrix (2000–2026)

In [4]:
def compute_strategy_analytics(series, spy_series, rf=0.025):
    daily_rets = series.pct_change().dropna()
    spy_rets = spy_series.pct_change().dropna()
    aligned = pd.concat([daily_rets, spy_rets], axis=1, join='inner').dropna()
    r_strat, r_spy = aligned.iloc[:, 0], aligned.iloc[:, 1]
    
    n_years = len(r_strat) / 252.0
    total_ret = (series.iloc[-1] / series.iloc[0]) - 1.0
    cagr = (series.iloc[-1] / series.iloc[0]) ** (1.0 / max(n_years, 0.1)) - 1.0
    ann_excess = (r_strat.mean() * 252.0) - rf
    ann_vol = r_strat.std() * np.sqrt(252.0)
    sharpe = ann_excess / ann_vol if ann_vol > 0 else 0.0
    downside_vol = r_strat[r_strat < 0].std() * np.sqrt(252.0)
    sortino = ann_excess / downside_vol if downside_vol > 0 else 0.0
    
    drawdown = (series - series.cummax()) / series.cummax()
    max_dd = drawdown.min()
    calmar = cagr / abs(max_dd) if abs(max_dd) > 0 else 0.0
    cov_matrix = np.cov(r_strat, r_spy)
    beta = cov_matrix[0, 1] / cov_matrix[1, 1] if cov_matrix[1, 1] > 0 else 1.0
    alpha = (cagr - rf) - beta * (((spy_series.iloc[-1] / spy_series.iloc[0]) ** (1.0 / max(n_years, 0.1)) - 1.0) - rf)
    
    return {
        'Total Return (%)': total_ret * 100.0,
        'CAGR (%)': cagr * 100.0,
        'Sharpe Ratio': sharpe,
        'Sortino Ratio': sortino,
        'Max Drawdown (%)': max_dd * 100.0,
        'Calmar Ratio': calmar,
        'Market Beta (β)': beta,
        'Jensen Alpha (α %)': alpha * 100.0
    }

eval_list = [
    ('Unified Engine (Concentrated Top 10 Alpha)', df_master_eval['Strategy_Unified_Concentrated_Top10']),
    ('Baseline Naive XGBoost (Top 100)', df_master_eval['Strategy_Baseline_XGBoost_Top100']),
    ('S&P 500 Index (SPY Benchmark)', df_master_eval['Benchmark_SPY_SP500'])
]

summary_stats = []
for name, s in eval_list:
    summary_stats.append({'Strategy / Benchmark': name, **compute_strategy_analytics(s, df_master_eval['Benchmark_SPY_SP500'])})

df_analytics_table = pd.DataFrame(summary_stats)
print("=== DEEP HISTORICAL (2000-2026) PERFORMANCE & RISK MATRIX ===")
df_analytics_table

=== DEEP HISTORICAL (2000-2026) PERFORMANCE & RISK MATRIX ===


,Strategy / Benchmark,Total Return (%),CAGR (%),Sharpe Ratio,Sortino Ratio,Max Drawdown (%),Calmar Ratio,Market Beta (β),Jensen Alpha (α %)
0,Unified Engine (Concentrated Top 10 Alpha),668.893443,8.130304,0.348523,0.448811,-58.894985,0.138047,0.809390,0.810425
1,Baseline Naive XGBoost (Top 100),2736.879372,13.677512,0.619056,0.786575,-52.556203,0.260245,0.989326,5.286120
2,S&P 500 Index (SPY Benchmark),731.460471,8.454957,0.389559,0.492366,-55.189469,0.153199,1.000000,0.000000


## 5. Historical Crisis Stress-Testing Table (2000–2026)

In [5]:
crises = [
    ("2000-2002 Dot-Com Bubble Crash", "2000-03-24", "2002-10-09"),
    ("2007-2009 Global Financial Crisis (GFC)", "2007-10-09", "2009-03-09"),
    ("2011 US Sovereign Debt Downgrade", "2011-05-02", "2011-10-03"),
    ("2018 Volmageddon & Q4 Sell-off", "2018-09-20", "2018-12-24"),
    ("2020 COVID-19 Flash Crash", "2020-02-19", "2020-03-23"),
    ("2022 Inflation & Rate-Hike Bear Market", "2022-01-03", "2022-10-12")
]

crisis_records = []
for c_name, start_d, end_d in crises:
    sub = df_master_eval[(df_master_eval['date'] >= start_d) & (df_master_eval['date'] <= end_d)]
    if not sub.empty:
        rec = {'Crisis Event': c_name}
        for col in ['Strategy_Unified_Concentrated_Top10', 'Strategy_Baseline_XGBoost_Top100', 'Benchmark_SPY_SP500']:
            s = sub[col]
            tot_drop = ((s.iloc[-1] / s.iloc[0]) - 1.0) * 100.0
            max_dd = (((s - s.cummax()) / s.cummax()).min()) * 100.0
            rec[f"{col}_Return"] = f"{tot_drop:+.1f}% (DD: {max_dd:.1f}%)"
        crisis_records.append(rec)

df_crisis_table = pd.DataFrame(crisis_records)
df_crisis_table.columns = ['Crisis Event', 'Unified Engine (Top 10)', 'Baseline XGBoost (Top 100)', 'S&P 500 (SPY)']
print("=== HISTORICAL CRISIS STRESS-TESTING RESULTS ===")
print(df_crisis_table.to_string(index=False))

=== HISTORICAL CRISIS STRESS-TESTING RESULTS ===
                           Crisis Event Unified Engine (Top 10) Baseline XGBoost (Top 100)       S&P 500 (SPY)
         2000-2002 Dot-Com Bubble Crash     -46.0% (DD: -53.1%)        -13.1% (DD: -33.6%) -45.4% (DD: -47.3%)
2007-2009 Global Financial Crisis (GFC)     -55.5% (DD: -58.9%)        -52.0% (DD: -52.6%) -55.2% (DD: -55.2%)
       2011 US Sovereign Debt Downgrade     -22.1% (DD: -22.4%)        -21.1% (DD: -21.4%) -18.5% (DD: -18.5%)
         2018 Volmageddon & Q4 Sell-off     -12.2% (DD: -12.4%)        -20.0% (DD: -20.1%) -19.3% (DD: -19.3%)
              2020 COVID-19 Flash Crash     -18.4% (DD: -19.7%)        -39.4% (DD: -39.4%) -33.7% (DD: -33.7%)
 2022 Inflation & Rate-Hike Bear Market     -15.6% (DD: -19.5%)        -18.6% (DD: -20.9%) -24.5% (DD: -24.5%)


## 6. 26-Year Interactive Equity Curves & Underwater Drawdowns (2000–2026)

In [6]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=('<b>Unified Alpha Engine (Top 10) vs. Baseline XGBoost (Top 100) vs. S&P 500 (SPY) (2000–2026)</b>',
                                    '<b>Underwater Drawdown Curves (%)</b>'))

colors = {
    'Strategy_Unified_Concentrated_Top10': '#00CC96',
    'Strategy_Baseline_XGBoost_Top100': '#AB63FA',
    'Benchmark_SPY_SP500': '#636EFA'
}

labels = {
    'Strategy_Unified_Concentrated_Top10': 'Unified Alpha Engine (Concentrated Top 10)',
    'Strategy_Baseline_XGBoost_Top100': 'Baseline Naive XGBoost (Top 100)',
    'Benchmark_SPY_SP500': 'S&P 500 (SPY Benchmark)'
}

for col, name in labels.items():
    s = df_master_eval[col]
    fig.add_trace(go.Scatter(
        x=df_master_eval['date'], y=s, name=name,
        line=dict(color=colors[col], width=3.5 if 'Unified' in col else 2.0)
    ), row=1, col=1)
    
    dd = ((s - s.cummax()) / s.cummax()) * 100.0
    fig.add_trace(go.Scatter(
        x=df_master_eval['date'], y=dd, name=f"{name} DD", showlegend=False,
        line=dict(color=colors[col], width=1.5)
    ), row=2, col=1)

fig.update_layout(
    template='plotly_dark', width=1150, height=750,
    title='<b>Deep Historical Performance Across 26+ Years (2000–2026)</b>',
    margin=dict(l=60, r=230, t=80, b=60),
    legend=dict(orientation='v', yanchor='top', y=1.0, xanchor='left', x=1.02, title=dict(text='<b>Strategy / Benchmark</b>'))
)
fig.show()

## 7. Annual Returns & Profit Breakdown by Year (2000–2026 Bar Chart)

In [7]:
df_master_eval['year'] = df_master_eval['date'].dt.year
strat_cols = ['Strategy_Unified_Concentrated_Top10', 'Strategy_Baseline_XGBoost_Top100', 'Benchmark_SPY_SP500']

annual_records = []
for y, grp in df_master_eval.groupby('year'):
    year_label = str(y)
    start_vals = grp.iloc[0][strat_cols]
    end_vals = grp.iloc[-1][strat_cols]
    rets = ((end_vals / start_vals) - 1.0) * 100.0
    record = {'Year': year_label}
    for col in strat_cols:
        record[labels[col]] = rets[col]
    annual_records.append(record)

df_annual_table = pd.DataFrame(annual_records)
print("=== 26-YEAR ANNUAL RETURNS BREAKDOWN BY YEAR (% PROFIT) ===")
print(df_annual_table.to_string(index=False))

fig_bar = go.Figure()
for col in strat_cols:
    lbl = labels[col]
    fig_bar.add_trace(go.Bar(
        x=df_annual_table['Year'],
        y=df_annual_table[lbl],
        name=lbl,
        marker_color=colors[col]
    ))

fig_bar.update_layout(
    template='plotly_dark',
    barmode='group',
    width=1150,
    height=600,
    title='<b>26-Year Annual Returns by Year (2000–2026): Unified Engine vs. Baseline vs. SPY</b>',
    xaxis=dict(title='<b>Trading Year</b>', tickangle=-45),
    yaxis=dict(title='<b>Annual Return (%)</b>', zeroline=True, zerolinewidth=1.5, zerolinecolor='gray'),
    margin=dict(l=60, r=230, t=80, b=60),
    legend=dict(orientation='v', yanchor='top', y=1.0, xanchor='left', x=1.02, title=dict(text='<b>Strategy / Benchmark</b>'))
)
fig_bar.show()

=== 26-YEAR ANNUAL RETURNS BREAKDOWN BY YEAR (% PROFIT) ===
Year  Unified Alpha Engine (Concentrated Top 10)  Baseline Naive XGBoost (Top 100)  S&P 500 (SPY Benchmark)
2000                                   -5.143918                          9.420907               -10.429517
2001                                    4.765032                         10.818202               -10.131550
2002                                  -12.986960                         -8.757203               -22.419488
2003                                   44.026199                         39.659705                24.184287
2004                                   16.627434                         19.016749                10.747757
2005                                    2.213547                         15.035311                 5.324944
2006                                   16.601194                         18.103834                13.842829
2007                                   16.539626                         15.

## 8. Export Results to Excel

In [8]:
output_deep_path = os.path.join(LOCAL_DATA_DIR, "deep_historical_2000_2026_simulation_poc.xlsx")
with pd.ExcelWriter(output_deep_path) as writer:
    df_master_eval.drop(columns=['year']).to_excel(writer, sheet_name='daily_equity_curves', index=False)
    df_analytics_table.to_excel(writer, sheet_name='summary_metrics', index=False)
    df_crisis_table.to_excel(writer, sheet_name='crisis_stress_testing', index=False)
    df_annual_table.to_excel(writer, sheet_name='annual_returns_by_year', index=False)

print(f"💾 Successfully exported 26-Year Deep Historical simulations to: {output_deep_path}")

💾 Successfully exported 26-Year Deep Historical simulations to: c:\Users\honza\Desktop\projects\stock-analysis\research\notebooks\algo-alpha-execution\data\fetched\deep_historical_2000_2026_simulation_poc.xlsx
